<a href="https://colab.research.google.com/github/JoshuneArriaga/Datos-Masivos/blob/main/Tarea_2_Datos_Masivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Descripción del conjunto de datos

Se utiliza el dump oficial de Wikipedia en ingles en formato XML, descargado directamente de [dumps.wikimedia.org](https://dumps.wikimedia.org/enwiki/latest/).

El dump completo contiene más de **6.8** millones de artículos y pesa 24.2 GB comprimido. Wikimedia divide este archivo en múltiples partes (multistream) para facilitar su descarga y procesamiento. Se utilizara para este proyecto solo la **parte 1** (~280 MB comprimido, ~41,000 artículos), que ya representa un volumen considerable de datos semi-estructurados.

###¿Por qué este conjunto de datos?

1. Contiene un numero masivo de datos, aun incluso una partición contiene decenas de artículos con texto completo.
2. Contiene datos semi estructurados
3. Los dumps de Wikipedia son uno de los datasets abiertos mas utilizados en la industria.
4. El mismo código puede escalar al dump completo de 24 GB en un clúster real de Spark.

### Estructura del XML de Wikipedia

```xml
<page>
  <title>Nombre del artículo</title>
  <ns>0</ns>                          <!-- 0 = artículo principal -->
  <id>12345</id>
  <revision>
    <id>67890</id>
    <timestamp>2025-12-15T10:30:00Z</timestamp>
    <contributor><username>Editor</username></contributor>
    <text bytes="5432">...contenido en wikitext...</text>
  </revision>
</page>
```

## 2. Instalación del entorno Spark en Google Colab

In [8]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#Check this site for the latest download link https://www.apache.org/dyn/closer.lua/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!wget -q https://archive.apache.org/dist/spark/spark-3.2.1/spark-3.2.1-bin-hadoop3.2.tgz
!tar xf spark-3.2.1-bin-hadoop3.2.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j
!pip install -q findspark pyspark py4j mwparserfromhell
import os
import sys

import findspark
findspark.init()
findspark.find()

import pyspark

from pyspark.sql import DataFrame, SparkSession
from typing import List
import pyspark.sql.types as T
import pyspark.sql.functions as F

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
82 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
from pyspark import SparkContext

sc = SparkContext('local[*]', 'Wikipedia RDDs')
print(f'SparkContext creado: {sc.master}')

SparkContext creado: local[*]


## 3. Parsing del XML y conversión a RDD

Descargamos la **parte 1** del dump multistream directamente de `dumps.wikimedia.org`.  

In [3]:
!mkdir -p data/wikipedia_src
!(cd data/wikipedia_src; wget -nc https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2)

DUMP_FILE = 'data/wikipedia_src/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2'
size_mb = os.path.getsize(DUMP_FILE) / (1024**2)
print(f'Archivo descargado: {size_mb:.1f} MB')

--2026-02-23 07:42:15--  https://dumps.wikimedia.org/enwiki/20260201/enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.71, 2620:0:861:3:208:80:154:71
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.71|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 295167616 (281M) [application/octet-stream]
Saving to: ‘enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2’

enwiki-20260201-pag 100%[===================>] 281.49M  4.81MB/s    in 59s     

2026-02-23 07:43:15 (4.75 MB/s) - ‘enwiki-20260201-pages-articles-multistream1.xml-p1p41242.bz2’ saved [295167616/295167616]

Archivo descargado: 281.5 MB


In [10]:
import bz2
import xml.etree.ElementTree as ET
import mwparserfromhell
import re
import time

def detect_namespace(filepath):
    """Detecta el namespace XML del dump automáticamente."""
    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for line in f:
            match = re.search(r'xmlns="([^"]+)"', line)
            if match:
                ns = '{' + match.group(1) + '}'
                print(f'Namespace: {ns}')
                return ns
    return ''

def wikitext_to_plain(wikicode):
    try:
        return mwparserfromhell.parse(wikicode).strip_code().strip()
    except:
        return wikicode

def parse_dump(filepath, max_articles=None):
    NS = detect_namespace(filepath)
    articles = []
    count = 0
    t0 = time.time()

    print('Parseando dump XML...')
    with bz2.open(filepath, 'rt', encoding='utf-8') as f:
        for event, elem in ET.iterparse(f, events=('end',)):
            if elem.tag != f'{NS}page':
                continue

            ns_tag = elem.find(f'{NS}ns')
            if ns_tag is None or ns_tag.text != '0':
                elem.clear()
                continue

            revision = elem.find(f'{NS}revision')
            if revision is None:
                elem.clear()
                continue

            text_elem = revision.find(f'{NS}text')
            wikitext = text_elem.text if (text_elem is not None and text_elem.text) else ''

            if wikitext.strip().upper().startswith('#REDIRECT'):
                elem.clear()
                continue

            title_elem = elem.find(f'{NS}title')
            id_elem    = elem.find(f'{NS}id')
            ts_elem    = revision.find(f'{NS}timestamp')

            plain_text = wikitext_to_plain(wikitext)
            categories = re.findall(r'\[\[Category:([^\]|]+)', wikitext)
            num_links  = len(re.findall(r'\[\[[^:][^\]]*\]\]', wikitext))
            num_refs   = len(re.findall(r'<ref[\s>]', wikitext))
            num_sections = len(re.findall(r'^==[^=]', wikitext, re.MULTILINE))

            articles.append({
                'id':             int(id_elem.text) if id_elem is not None else 0,
                'title':          title_elem.text if title_elem is not None else '',
                'word_count':     len(plain_text.split()),
                'text_length':    len(plain_text),
                'timestamp':      ts_elem.text if ts_elem is not None else '',
                'num_categories': len(categories),
                'num_links':      num_links,
                'num_references': num_refs,
                'num_sections':   num_sections,
            })

            count += 1
            if count % 5000 == 0:
                print(f'  {count:>6,} artículos  ({time.time()-t0:.0f}s)')
            elem.clear()

            if max_articles and count >= max_articles:
                break

    print(f'Terminado: {count:,} artículos  ({time.time()-t0:.0f}s)')
    return articles

In [11]:
articles = parse_dump(DUMP_FILE, max_articles=30_000)

Namespace: {http://www.mediawiki.org/xml/export-0.11/}
Parseando dump XML...
   5,000 artículos  (299s)
  10,000 artículos  (613s)
  15,000 artículos  (920s)
  20,000 artículos  (1149s)
Terminado: 21,009 artículos  (1180s)


In [12]:
# Convertir a RDD con parallelize
rdd = sc.parallelize(articles)

print(f'Tipo: {type(rdd)}')
print(f'Número de particiones: {rdd.getNumPartitions()}')
print(f'Conteo: {rdd.count()}')

Tipo: <class 'pyspark.core.rdd.RDD'>
Número de particiones: 2
Conteo: 21009


In [13]:
# Ver los primeros 3 elementos del RDD
for art in rdd.take(3):
    print(art)
    print()

{'id': 12, 'title': 'Anarchism', 'word_count': 6818, 'text_length': 47597, 'timestamp': '2026-01-27T23:56:33Z', 'num_categories': 13, 'num_links': 648, 'num_references': 14, 'num_sections': 9}

{'id': 39, 'title': 'Albedo', 'word_count': 4487, 'text_length': 29053, 'timestamp': '2026-01-25T05:49:13Z', 'num_categories': 10, 'num_links': 181, 'num_references': 138, 'num_sections': 7}

{'id': 290, 'title': 'A', 'word_count': 2065, 'text_length': 13017, 'timestamp': '2026-01-31T15:07:02Z', 'num_categories': 2, 'num_links': 322, 'num_references': 24, 'num_sections': 10}



## 4. Transformaciones con `map`

Extraemos columnas individuales del RDD de diccionarios usando `map`.

In [14]:
# Extraer solo el conteo de palabras de cada artículo
rdd_words = rdd.map(lambda art: art['word_count'])
rdd_words.take(10)

[6818, 4487, 2065, 13600, 8867, 11906, 10549, 2195, 7364, 12074]

In [15]:
# calcular palabras por sección para cada artículo
rdd_words_per_section = rdd.map(
    lambda art: (
        art['title'],
        round(art['word_count'] / art['num_sections'], 1) if art['num_sections'] > 0 else float(art['word_count'])
    )
)
rdd_words_per_section.take(5)

[('Anarchism', 757.6),
 ('Albedo', 641.0),
 ('A', 206.5),
 ('Alabama', 906.7),
 ('Achilles', 806.1)]

## 5. Transformaciones con `filter`

In [16]:
# Filtrar artículos con más de 2000 palabras
rdd_largos = rdd.filter(lambda art: art['word_count'] > 2000)
print(f'Artículos con > 2000 palabras: {rdd_largos.count()}')

# Mostrar títulos de los primeros 10
rdd_largos.map(lambda art: (art['title'], art['word_count'])).take(10)

Artículos con > 2000 palabras: 11854


[('Anarchism', 6818),
 ('Albedo', 4487),
 ('A', 2065),
 ('Alabama', 13600),
 ('Achilles', 8867),
 ('Abraham Lincoln', 11906),
 ('Aristotle', 10549),
 ('An American in Paris', 2195),
 ('Academy Award for Best Production Design', 7364),
 ('Academy Awards', 12074)]

In [17]:
# Filtrar artículos que no tienen ninguna referencia
rdd_sin_refs = rdd.filter(lambda art: art['num_references'] == 0)
print(f'Artículos sin referencias: {rdd_sin_refs.count()} de {rdd.count()}')
rdd_sin_refs.map(lambda art: (art['title'], art['word_count'])).take(5)

Artículos sin referencias: 2045 de 21009


[('Alien', 908),
 ('Austin', 415),
 ('Ada', 514),
 ('Aberdeen (disambiguation)', 770),
 ('Argument (disambiguation)', 233)]

---
## 6. Estadísticas descriptivas con RDDs

In [18]:
rdd_wc = rdd.map(lambda art: art['word_count'])

print('=== Estadísticas de word_count ===')
print('Conteo:', rdd_wc.count())
print('Media:', round(rdd_wc.mean(), 2))
print('Desv. estándar:', round(rdd_wc.stdev(), 2))
print('Mínimo:', rdd_wc.min())
print('Máximo:', rdd_wc.max())
print('Suma:', rdd_wc.sum())

=== Estadísticas de word_count ===
Conteo: 21009
Media: 3722.64
Desv. estándar: 3807.92
Mínimo: 0
Máximo: 36136
Suma: 78209002


In [19]:

rdd_refs = rdd.map(lambda art: art['num_references'])

print('=== Estadísticas de num_references ===')
print('Conteo:', rdd_refs.count())
print('Media:', round(rdd_refs.mean(), 2))
print('Desv. estándar:', round(rdd_refs.stdev(), 2))
print('Mínimo:', rdd_refs.min())
print('Máximo:', rdd_refs.max())

=== Estadísticas de num_references ===
Conteo: 21009
Media: 71.72
Desv. estándar: 99.34
Mínimo: 0
Máximo: 1254
